In [ ]:
!pip install transformers[torch] datasets evaluate rouge_score nltk accelerate -q

import numpy as np
import nltk
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
import evaluate

nltk.download("punkt")

print("Đang tải dữ liệu PubMedQA...")
# pqa_artificial rất lớn (211k mẫu), mình sẽ lấy một phần để chạy thử nghiệm 
# Nếu bạn muốn train toàn bộ, hãy bỏ .select(range(...))
raw_dataset = load_dataset("pubmed_qa", "pqa_artificial", split="train")
raw_dataset = raw_dataset.shuffle(seed=42)


# Chia 80% train, 20% validation
dataset = raw_dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]


# 2. Load Tokenizer và Model (FLAN-T5-small)
# ---------------------------------------------------------
model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)


# 3. Tiền xử lý dữ liệu (Preprocessing)
# ---------------------------------------------------------
def preprocess_function(examples):
    # Tạo Prompt
    inputs = []
    for i in range(len(examples["question"])):
        ctx = " ".join(examples["context"][i]["contexts"])
        ques = examples["question"][i]
        prompt = f"Answer the medical question with yes, no or maybe.\nQuestion: {ques}\nContext: {ctx}"
        inputs.append(prompt)
    # Tokenize Input
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Tokenize Output (Labels)
    labels = tokenizer(text_target=examples["final_decision"], max_length=8, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


print("Đang tiền xử lý dữ liệu...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)



# 4. Định nghĩa Hàm tính Metrics (Accuracy, F1, ROUGE, BLEU, EM)
# ---------------------------------------------------------
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Giải mã predictions (từ ID sang chữ)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Giải mã labels (thay thế -100 vốn dùng để bỏ qua loss)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Làm sạch văn bản để so sánh
    decoded_preds = [pred.strip().lower() for pred in decoded_preds]
    decoded_labels = [label.strip().lower() for label in decoded_labels]

    # --- Tính Exact Match ---
    em = np.mean([p == l for p, l in zip(decoded_preds, decoded_labels)])

    # --- Chuẩn bị nhãn số để tính Acc/F1/Prec/Rec ---
    label_map = {"yes": 1, "no": 0, "maybe": 2}
    def to_num(txt): return label_map.get(txt, -1) # Trả về -1 nếu sinh từ lạ

    num_preds = [to_num(p) for p in decoded_preds]
    num_labels = [to_num(l) for l in decoded_labels]

    # --- Tính toán các chỉ số ---
    results = {}
    results["accuracy"] = acc_metric.compute(predictions=num_preds, references=num_labels)["accuracy"]
    results["f1_macro"] = f1_metric.compute(predictions=num_preds, references=num_labels, average="macro")["f1"]
    results["precision"] = precision_metric.compute(predictions=num_preds, references=num_labels, average="macro", zero_division=0)["precision"]
    results["recall"] = recall_metric.compute(predictions=num_preds, references=num_labels, average="macro", zero_division=0)["recall"]
    
    # Rouge & Bleu
    rouge_res = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    results["rougeL"] = rouge_res["rougeL"]
    
    bleu_res = bleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    results["bleu"] = bleu_res["bleu"]
    
    results["exact_match"] = em

    return results

#5. Cấu hình Huấn luyện (Training Arguments)
# ---------------------------------------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan_t5_pubmedqa",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True, # Quan trọng để tính metrics
    fp16=True,                  # Bật nếu dùng GPU (nhanh hơn)
    logging_steps=100,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 6. Chạy Huấn luyện và Đánh giá
# ---------------------------------------------------------
print("Bắt đầu huấn luyện...")
trainer.train()

print("\nĐang đánh giá cuối cùng...")
final_results = trainer.evaluate()

print("\nKẾT QUẢ CUỐI CÙNG:")
for key, value in final_results.items():
    print(f"{key}: {value:.4f}")

# Lưu model và tokenizer
model.save_pretrained("./flan_t5_pubmedqa_trained_full_dataset")
tokenizer.save_pretrained("./flan_t5_pubmedqa_trained_full_dataset")

!zip -r flan_t5_pubmedqa_trained_full_dataset.zip flan_t5_pubmedqa_trained_full_dataset